In [29]:
from dotenv import load_dotenv

load_dotenv()

True

In [30]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.1,
)

In [31]:
examples = [
    {
        "movie": "Top Gun",
        "answer": "🛩️👨‍✈️🔥",
    },
    {
        "movie": "The Godfather",
        "answer": "👨‍👨‍👦🔫🍝",
    },
    {
        "movie": "Titanic",
        "answer": "🚢🌊💔",
    },
    {
        "movie": "Jurassic Park",
        "answer": "🦖🏝️🚙",
    },
    {
        "movie": "Harry Potter",
        "answer": "🧙‍♂️⚡🏰",
    },
]

In [32]:
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "Represent the movie {movie} with three emojis."),
        ("ai", "{answer}")
    ]
)

In [33]:
fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

In [34]:
store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [35]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a movie emoji expert.

When the user gives you a movie title, reply with exactly three emojis that represent the movie.
Do not write any words.
Do not add explanations.
Do not use more or fewer than three emojis.

If the user asks about the conversation history, answer the question based on the previous messages.
""",
        ),
        MessagesPlaceholder(variable_name="history"),
        fewshot_prompt,
        ("human", "{question}"),
    ]
)

In [36]:
chain = prompt | llm

chain_with_memory = RunnableWithMessageHistory(
    runnable=chain,
    get_session_history=get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)

/Users/browndays/VSCode/langchain/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [39]:
config = {"configurable": {"session_id": "movie_emoji_session"}}

response1 = chain_with_memory.invoke(
    {"question": "The Matrix"},
    config=config,
)

print("The Matrix:", response1.content)

response2 = chain_with_memory.invoke(
    {"question": "Finding Nemo"},
    config=config,
)

print("Finding Nemo:", response2.content)

The Matrix: 💊💻🕶️
Finding Nemo: 🐠🐢🐋


In [38]:
response3 = chain_with_memory.invoke(
    {"question": "what was the first movie I asked you about?"},
    config=config,
)

print(response3.content)

The first movie you asked me about was "The Matrix".
